# 03_Train.ipynb: Walk-Forward, Calibration & Model Training
Tuân thủ nghiêm ngặt AGENTS.md và PROCESS.md trong workspace hiện tại.
Version 2 Stage 2 Training, Prequential Evaluation & Offline Threshold Tuning Notebook.

- **Mục tiêu:** Huấn luyện Walk-Forward và đánh giá prequential OOS cho 2 nhánh:
  1. **Primary Quality-Risk Forecast (Branch A):** Dự báo nguy cơ chất lượng tương lai $[t+L, t+L+H)$ với $L=6\text{h}$, $D=6\text{h}$, $W=168\text{h}$, cadence $6\text{h}$, $H \in [6\text{h}, 12\text{h}, 24\text{h}]$, ngưỡng tỷ lệ lỗi $q=0.10$, min_event_samples=5.
  2. **Secondary QC Inspection Triage (Branch B):** Phân loại rủi ro mẫu raw đo kiểm ngay khi vừa xuất xưởng trước khi có kết quả QC ($t+6\text{h}$).
- **Kỷ luật:** Mọi ước lượng tiền xử lý và lựa chọn đặc trưng chỉ fit trong từng cửa sổ huấn luyện rolling 56 ngày; purged hoàn toàn tương lai; tối ưu hóa ngưỡng $\tau$ chỉ dùng điểm số OOS quá khứ đã trưởng thành.
- **Tập Holdout:** Không đụng đến tập Test/Holdout trong quá trình lựa chọn thuật toán; mọi quyết định chọn mô hình chỉ dựa trên tập Development (< 2008-10-02 20:54).

In [1]:
# 1. Environment Bootstrap & Ingestion of 02_Preprocessing Definitions (Tagged: v2_training_helpers)
import os
import sys
import hashlib
import json
import pathlib
import time
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier,
    ExtraTreesRegressor,
    ExtraTreesClassifier,
    RandomForestClassifier
)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, f_classif, f_regression
from sklearn.preprocessing import RobustScaler
import joblib

# Locate PROJECT_ROOT dynamically by walking up cwd parents
p = pathlib.Path('.').resolve()
PROJECT_ROOT = None
for parent in [p] + list(p.parents):
    if (parent / 'AGENTS.md').exists() and (parent / 'data' / 'secom').exists():
        PROJECT_ROOT = parent
        break
assert PROJECT_ROOT is not None, 'PROJECT_ROOT could not be identified!'
VERSION_DIR = PROJECT_ROOT / 'Version 2'

print(f'[BOOTSTRAP] PROJECT_ROOT: {PROJECT_ROOT}')
print(f'[BOOTSTRAP] VERSION_DIR:  {VERSION_DIR}')

# Load 02_Preprocessing definitions excluding test cells
nb_02_path = VERSION_DIR / '02_Preprocessing.ipynb'
assert nb_02_path.exists(), f'02_Preprocessing.ipynb not found at {nb_02_path}'

with open(nb_02_path, 'r', encoding='utf-8') as f:
    nb_02_json = json.load(f)

globs_02 = {}
loaded_cells_count = 0
for cell in nb_02_json.get('cells', []):
    if cell.get('cell_type') == 'code':
        src = ''.join(cell.get('source', []))
        if 'run_synthetic_tests' not in src:
            exec(src, globs_02)
            loaded_cells_count += 1

# Expose definitions to global scope
for k, v in globs_02.items():
    if not k.startswith('_'):
        globals()[k] = v

print(f'[BOOTSTRAP] Successfully imported {loaded_cells_count} definition code cells from 02_Preprocessing.ipynb')
print('[BOOTSTRAP] Available functions: load_raw, make_features, build_targets, training_maturity_gate, interval_metrics, row_metrics, capacity_threshold, qc_threshold')

[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2
[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2
[BOOTSTRAP] Successfully imported 5 definition code cells from 02_Preprocessing.ipynb
[BOOTSTRAP] Available functions: load_raw, make_features, build_targets, training_maturity_gate, interval_metrics, row_metrics, capacity_threshold, qc_threshold


## 2. API Contract & Helper Specifications for 04_Metrics.ipynb
Các hàm helper được đánh dấu thẻ metadata `tags: ["v2_training_helpers"]` tuân thủ nghiêm ngặt hợp đồng:
- **`fit_model_state(raw, X_train, y_train, candidate, task_type='forecast', weights=None, fit_time=None, max_train_origin=None, delay_h=6)`**:
  Huấn luyện mô hình và tiền xử lý cục bộ: Raw sensor quality filter + SimpleImputer + VarianceThreshold + Correlation-pruned Univariate feature selector + RobustScaler. Trả về state dict chuẩn của scikit-learn, ghi nhận đầy đủ provenance (`fit_time`, `last_train_label_available`, `raw_filter_max_timestamp`, `n_train`). Không chứa custom classes hay unpicklable lambdas.
- **`predict_score(state, X_pred)`**:
  Dự báo continuous OOS scores trên ma trận đặc trưng mới.
- **`forecast_stream(raw, X, avail_meta, targets, candidate, config)`**:
  Trình mô phỏng walk-forward prequential theo trình tự thời gian cho Branch A. Tái huấn luyện daily (>= 24h), cửa sổ rolling 56 ngày, kiểm tra nghiêm ngặt `target_available <= t` và loại bỏ hoàn toàn các target chưa xác định (`event.notna()` hoặc `rate.notna()`). Trả về `(score_df, last_state, fit_trace)`.
- **`apply_forecast_policy(score_df, raw, targets, budget, lookback_days, eval_span, disjoint_spans=None)`**:
  Tái hiện chính sách cảnh báo trực tuyến với `capacity_threshold`, đánh giá nghiêm ngặt trên các khoảng cửa sổ $[t+L, t+L+H)$ nằm trọn vẹn trong `eval_span`. Trả về `(warning_df, summary_metrics, fold_metrics)`.
- **`qc_stream(raw, candidate, config)`**:
  Trình mô phỏng walk-forward prequential cấp raw sample cho Branch B. Tái huấn luyện daily với các nhãn raw đã trưởng thành (`timestamp + 6h <= t`). Trả về `(score_df, last_state, fit_trace)`.
- **`apply_qc_policy(score_df, raw, budget, lookback_days=14, report_spans=None)`**:
  Mô phỏng hạn ngạch kiểm định trực tuyến theo khối báo cáo calendar, kiểm soát trần workload $W \le \text{budget}$ tại mọi dòng đến. Trả về `(warning_df, summary_metrics, fold_metrics)`.

In [2]:
# 3. Core Preprocessing, Feature Selection & Estimator Fitter (Tagged: v2_training_helpers)
def fit_model_state(raw, X_train, y_train, candidate, task_type='forecast', weights=None, fit_time=None, max_train_origin=None, delay_h=6):
    """
    Fit standard estimator and causal feature pipeline inside training fold.
    Never leaks future observations or global dataset statistics.
    Returns standard dictionary state compatible with joblib (no custom classes/lambdas).
    Provenance tracked: fit_time, last_train_label_available, raw_filter_max_timestamp, n_train.
    """
    fit_time = pd.Timestamp(fit_time) if fit_time is not None else None
    
    # 1. Fallback / Baseline trivial states
    if candidate == 'never_alert':
        return {
            'candidate': candidate, 'type': 'never_alert', 'fit_time': fit_time,
            'last_train_label_available': fit_time, 'raw_filter_max_timestamp': fit_time,
            'n_train': 0, 'train_positive': 0, 'selected_feature_names': []
        }
    if candidate.startswith('baseline_') or candidate in ['fail_rate24', 'fail_rate72', 'fail_rate168', 'past_qc_ewma24']:
        col = candidate.replace('baseline_', '')
        return {
            'candidate': candidate, 'type': 'baseline', 'col': col, 'fit_time': fit_time,
            'last_train_label_available': fit_time, 'raw_filter_max_timestamp': fit_time,
            'n_train': 0, 'train_positive': 0, 'selected_feature_names': [col]
        }

    # 2. Quality filter on raw sensor channels
    # Only fit on raw observations in union of TRAIN history windows
    t_filt_start = time.time()
    sensor_cols = [c for c in raw.columns if c.startswith('sensor_')]
    if task_type == 'forecast':
        assert max_train_origin is not None, 'max_train_origin required for forecast task raw filter'
        min_train_origin = X_train.index.min()
        raw_filt_max = max_train_origin - pd.Timedelta(hours=delay_h)
        assert raw_filt_max <= max_train_origin - pd.Timedelta(hours=delay_h), 'Raw filter max exceeds max_train_origin - delay_h!'
        raw_win = raw[(raw['timestamp'] >= min_train_origin - pd.Timedelta(hours=168)) & 
                      (raw['timestamp'] <= raw_filt_max)]
    else:
        # Branch B: raw rows in X_train
        raw_win = raw.loc[X_train.index]
        raw_filt_max = raw_win['timestamp'].max() if len(raw_win) > 0 else fit_time
        
    sub_sensors = raw_win[sensor_cols]
    n_unique = sub_sensors.nunique(dropna=True)
    missing_ratio = sub_sensors.isna().mean()
    passed_sensors = set(sub_sensors.columns[(n_unique > 1) & (missing_ratio <= 0.50)])
    filt_duration = time.time() - t_filt_start

    # Determine permitted feature columns in X_train
    all_cols = list(X_train.columns)
    permitted_cols = []
    for c in all_cols:
        if not c.startswith('sensor_'):
            permitted_cols.append(c)
        else:
            base_s = c.split('__')[0]
            if '__missing_fraction' in c or base_s in passed_sensors:
                permitted_cols.append(c)

    # Filter candidate features if history-only model
    if 'history' in candidate:
        permitted_cols = [c for c in permitted_cols if not c.startswith('sensor_')]

    # Check for empty permitted columns -> declare prior fallback
    if len(permitted_cols) == 0:
        prior_val = float(y_train.mean()) if len(y_train) > 0 else 0.05
        return {
            'candidate': candidate, 'type': 'fallback', 'fallback_val': prior_val,
            'fit_time': fit_time, 'last_train_label_available': fit_time,
            'raw_filter_max_timestamp': raw_filt_max, 'n_train': len(X_train),
            'train_positive': int(y_train.sum()) if hasattr(y_train, 'sum') else 0,
            'selected_feature_names': []
        }

    X_tr_perm = X_train[permitted_cols].values

    # 3. Imputation and Variance Filtering
    imputer = SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)
    X_imp = imputer.fit_transform(X_tr_perm)
    imputed_feature_names = imputer.get_feature_names_out(permitted_cols)
    assert len(imputed_feature_names) == X_imp.shape[1], 'Imputer feature names count mismatch!'

    var_thresh = VarianceThreshold(threshold=0.0)
    X_var = var_thresh.fit_transform(X_imp)
    var_support = var_thresh.get_support()
    names_after_var = np.array(imputed_feature_names)[var_support]

    if X_var.shape[1] == 0:
        prior_val = float(y_train.mean()) if len(y_train) > 0 else 0.05
        return {
            'candidate': candidate, 'type': 'fallback', 'fallback_val': prior_val,
            'fit_time': fit_time, 'last_train_label_available': fit_time,
            'raw_filter_max_timestamp': raw_filt_max, 'n_train': len(X_train),
            'train_positive': int(y_train.sum()) if hasattr(y_train, 'sum') else 0,
            'selected_feature_names': []
        }

    # 4. Supervised Feature Selection (Univariate + Correlation Pruning)
    k = 0
    if '16' in candidate:
        k = 16
    elif '24' in candidate:
        k = 24
    elif '64' in candidate:
        k = 64
    elif 'k1' in candidate:
        k = 1
    elif 'k8' in candidate:
        k = 8

    if k > 0 and 'history' not in candidate and X_var.shape[1] > k:
        if candidate.startswith('lr_') or task_type == 'qc':
            # Classification target
            f_scores, _ = f_classif(X_var, y_train.values)
        else:
            # Regression rate target
            f_scores, _ = f_regression(X_var, y_train.values)
        f_scores = np.nan_to_num(f_scores, nan=0.0)
        pool_size = min(4 * k, X_var.shape[1])
        pool_idx = np.argsort(f_scores)[::-1][:pool_size]

        selected_idx = []
        for idx in pool_idx:
            col_v = X_var[:, idx]
            is_corr = False
            for s_idx in selected_idx:
                r = np.corrcoef(col_v, X_var[:, s_idx])[0, 1]
                if np.isnan(r) or abs(r) >= 0.98:
                    is_corr = True
                    break
            if not is_corr:
                selected_idx.append(idx)
                if len(selected_idx) == k:
                    break
        selected_idx = sorted(selected_idx)
    else:
        selected_idx = list(range(X_var.shape[1]))

    X_sel = X_var[:, selected_idx]
    selected_feature_names = list(names_after_var[selected_idx])

    # 5. Robust Scaling
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_sel)

    # 6. Fit Standard Model Object
    # Primary Forecast Models (Branch A)
    if candidate == 'lr_history' or candidate == 'lr_sensor16':
        model = LogisticRegression(C=0.1, class_weight='balanced', max_iter=500, random_state=42)
        model.fit(X_scaled, y_train.values)
    elif candidate == 'ridge_history':
        model = Ridge(alpha=10.0, random_state=42)
        model.fit(X_scaled, y_train.values, sample_weight=weights)
    elif candidate == 'ridge_sensor16':
        model = Ridge(alpha=100.0, random_state=42)
        model.fit(X_scaled, y_train.values, sample_weight=weights)
    elif candidate == 'hgb_poisson24':
        model = HistGradientBoostingRegressor(
            loss='poisson', max_iter=60, max_leaf_nodes=7, min_samples_leaf=10,
            l2_regularization=10.0, early_stopping=False, random_state=42
        )
        model.fit(X_scaled, y_train.values, sample_weight=weights)
    elif candidate == 'et_rate24':
        model = ExtraTreesRegressor(
            n_estimators=100, max_depth=4, min_samples_leaf=8, n_jobs=2, random_state=42
        )
        model.fit(X_scaled, y_train.values, sample_weight=weights)

    # Secondary QC Triage Models (Branch B)
    elif candidate in ['lr_qc_k1', 'lr_qc_k8', 'lr_qc_k24', 'balanced_lr_k1', 'balanced_lr_k8', 'balanced_lr_k24']:
        model = LogisticRegression(C=0.1, class_weight='balanced', max_iter=500, random_state=42)
        model.fit(X_scaled, y_train.values)
    elif candidate in ['lr_qc_k24_c01', 'balanced_lr_k24_c01']:
        model = LogisticRegression(C=0.01, class_weight='balanced', max_iter=500, random_state=42)
        model.fit(X_scaled, y_train.values)
    elif candidate in ['rf_qc', 'balanced_rf']:
        model = RandomForestClassifier(
            n_estimators=200, max_depth=4, min_samples_leaf=5, class_weight='balanced', n_jobs=2, random_state=42
        )
        model.fit(X_scaled, y_train.values)
    elif candidate in ['hgb_qc', 'balanced_hgb']:
        model = HistGradientBoostingClassifier(
            max_iter=60, max_leaf_nodes=7, min_samples_leaf=10, l2_regularization=10.0,
            class_weight='balanced', early_stopping=False, random_state=42
        )
        model.fit(X_scaled, y_train.values)
    elif candidate in ['et_qc_k64', 'balanced_et_k64']:
        model = ExtraTreesClassifier(
            n_estimators=200, max_depth=6, min_samples_leaf=3, class_weight='balanced', n_jobs=2, random_state=42
        )
        model.fit(X_scaled, y_train.values)
    elif candidate in ['lr_qc_k24_half_odd', 'lr_qc_k24_half_rand']:
        model = LogisticRegression(C=0.1, class_weight=None, max_iter=500, random_state=42)
        model.fit(X_scaled, y_train.values)
    else:
        raise ValueError(f'Unknown candidate: {candidate}')

    return {
        'candidate': candidate,
        'type': 'learned',
        'task_type': task_type,
        'permitted_cols': permitted_cols,
        'imputer': imputer,
        'var_thresh': var_thresh,
        'selected_idx': selected_idx,
        'selected_feature_names': selected_feature_names,
        'scaler': scaler,
        'model': model,
        'fit_time': fit_time,
        'raw_filter_max_timestamp': raw_filt_max,
        'raw_filter_duration_s': filt_duration,
        'n_train': len(X_train),
        'train_positive': int(y_train.sum()) if hasattr(y_train, 'sum') else np.nan
    }

def predict_score(state, X_pred):
    """
    Produce out-of-sample continuous decision scores from fitted state.
    Classifiers return positive class probability or decision function; regressors return clipped [0, 1] rate.
    """
    stype = state.get('type')
    if stype == 'never_alert':
        return pd.Series(-np.inf, index=X_pred.index)
    if stype == 'baseline':
        col = state['col']
        return X_pred[col].fillna(0.0).astype(float)
    if stype == 'fallback':
        return pd.Series(float(state['fallback_val']), index=X_pred.index)
        
    # Standard learned pipeline transform
    permitted_cols = state['permitted_cols']
    X_sub = X_pred[permitted_cols].values
    X_imp = state['imputer'].transform(X_sub)
    X_var = state['var_thresh'].transform(X_imp)
    X_sel = X_var[:, state['selected_idx']]
    X_scaled = state['scaler'].transform(X_sel)
    
    model = state['model']
    if hasattr(model, 'predict_proba'):
        preds = model.predict_proba(X_scaled)[:, 1]
    elif hasattr(model, 'decision_function'):
        preds = model.decision_function(X_scaled)
    elif hasattr(model, 'predict'):
        preds = np.clip(model.predict(X_scaled), 0.0, 1.0)
    else:
        raise TypeError('Model object has neither predict_proba, decision_function, nor predict')
        
    return pd.Series(preds, index=X_pred.index)

In [3]:
# 4. Prequential Walk-Forward Simulators & Online Policy Replays (Tagged: v2_training_helpers)
def forecast_stream(raw, X, avail_meta, targets, candidate, config):
    """
    Chronological walk-forward simulator with daily refitting (>= 24h) and 56-day rolling train window.
    Strictly enforces prequential maturity gating: targets completed and available before origin (target_available <= t).
    Eliminates undefined/sparse target corruptions (event.notna() for classifiers, rate.notna() for regressors).
    Produces out-of-sample continuous score record for all prediction origins.
    """
    origins = X.index
    score_records = []
    fit_trace = []
    last_state = None
    last_fit_time = None
    
    delay_h = config.get('delay_h', 6)
    rolling_days = config.get('rolling_days', 56)
    refit_cadence_h = config.get('refit_cadence_h', 24)
    min_train_origins = config.get('min_train_origins', 40)
    
    # Pre-identify baseline / never_alert
    is_baseline = candidate == 'never_alert' or candidate in ['fail_rate24', 'fail_rate72', 'fail_rate168', 'past_qc_ewma24']
    is_classifier = candidate.startswith('lr_')
    is_poisson = 'poisson' in candidate
    
    for orig in origins:
        orig = pd.Timestamp(orig)
        
        # Verify feature availability assertion
        orig_avail = avail_meta.loc[orig, 'max_feature_available']
        if pd.notna(orig_avail):
            assert orig_avail <= orig, f'Future feature leak at origin {orig}: {orig_avail}'
            
        if is_baseline:
            sc = -np.inf if candidate == 'never_alert' else float(X.loc[orig, candidate])
            score_records.append({
                'origin': orig,
                'target_start': targets.loc[orig, 'target_start'],
                'target_end': targets.loc[orig, 'target_end'],
                'target_available': targets.loc[orig, 'target_available'],
                'score': sc,
                'fit_time': orig,
                'last_train_label_available': orig,
                'train_n': 0,
                'feature_max_available': orig_avail
            })
            continue
            
        # Check refit condition (last_state is None checked BEFORE date delta to avoid overflow)
        refit_due = (last_state is None) or ((orig - last_fit_time).total_seconds() >= refit_cadence_h * 3600.0)
        
        if refit_due:
            train_start = orig - pd.Timedelta(days=rolling_days)
            m_eligible = (targets['origin'] >= train_start) & (targets['target_available'] <= orig)
            mature_meta = targets.loc[m_eligible]
            
            # Assert that no future targets or un-matured windows enter training
            if len(mature_meta) > 0:
                assert (mature_meta['target_available'] <= orig).all(), f'Future target leakage at {orig}'
                assert (mature_meta['target_end'] <= orig - pd.Timedelta(hours=delay_h)).all(), f'Target end overlap at {orig}'
                
            # Filter defined labels only (never fill undefined targets with 0)
            if is_classifier:
                mature_meta = mature_meta[mature_meta['event'].notna()]
            else:
                mature_meta = mature_meta[mature_meta['rate'].notna()]
                
            valid_train = False
            if len(mature_meta) >= min_train_origins:
                if is_classifier:
                    ev_counts = mature_meta['event'].value_counts()
                    valid_train = (len(ev_counts) >= 2) and (ev_counts.min() >= 1)
                elif is_poisson:
                    valid_train = mature_meta['rate'].sum() > 0
                else:
                    valid_train = True
                    
            if valid_train:
                tr_idx = mature_meta.index
                X_tr = X.loc[tr_idx]
                max_tr_orig = tr_idx.max()
                actual_label_avail_max = mature_meta['target_available'].max()
                
                if is_classifier:
                    y_tr = mature_meta.loc[tr_idx, 'event'].astype(int)
                    w_tr = None
                else:
                    y_tr = mature_meta.loc[tr_idx, 'rate'].astype(float)
                    w_tr = np.clip(mature_meta.loc[tr_idx, 'n_future'].values, 1, 50)
                    
                state = fit_model_state(
                    raw=raw, X_train=X_tr, y_train=y_tr, candidate=candidate,
                    task_type='forecast', weights=w_tr, fit_time=orig,
                    max_train_origin=max_tr_orig, delay_h=delay_h
                )
                state['last_train_label_available'] = actual_label_avail_max
                last_state = state
                last_fit_time = orig
                train_n = len(mature_meta)
                fit_trace.append({
                    'fit_time': orig, 'train_n': train_n,
                    'last_label_avail': actual_label_avail_max, 'status': 'fit'
                })
            else:
                past_rate_mean = float(mature_meta['rate'].mean()) if (len(mature_meta) > 0 and mature_meta['rate'].notna().any()) else 0.05
                actual_label_avail_max = mature_meta['target_available'].max() if len(mature_meta) > 0 else pd.NaT
                last_state = {
                    'candidate': candidate, 'type': 'fallback', 'fallback_val': past_rate_mean,
                    'fit_time': orig, 'last_train_label_available': actual_label_avail_max,
                    'n_train': len(mature_meta)
                }
                last_fit_time = orig
                train_n = len(mature_meta)
                fit_trace.append({
                    'fit_time': orig, 'train_n': train_n,
                    'last_label_avail': actual_label_avail_max, 'status': 'fallback'
                })

        # Predict continuous OOS score for current origin
        X_cur = X.loc[[orig]]
        sc = float(predict_score(last_state, X_cur).iloc[0])
        score_records.append({
            'origin': orig,
            'target_start': targets.loc[orig, 'target_start'],
            'target_end': targets.loc[orig, 'target_end'],
            'target_available': targets.loc[orig, 'target_available'],
            'score': sc,
            'fit_time': last_fit_time,
            'last_train_label_available': last_state.get('last_train_label_available'),
            'train_n': last_state.get('n_train', 0),
            'feature_max_available': orig_avail
        })
        
    score_df = pd.DataFrame(score_records).set_index('origin', drop=False)
    return score_df, last_state, fit_trace

def apply_forecast_policy(score_df, raw, targets, budget, lookback_days, eval_span, disjoint_spans=None):
    """
    Replay online capacity thresholding using past matured OOS scores produced up to prediction origin.
    Only evaluates forecast windows strictly contained in eval_span: target_start >= eval_start and target_end <= eval_end.
    Calculates unified metrics over eval_span and individual disjoint folds with clipped warning masks.
    """
    eval_start, eval_end = pd.Timestamp(eval_span[0]), pd.Timestamp(eval_span[1])
    
    # Identify in-eval origins whose targets are fully contained in global eval span
    m_eval = (targets['target_start'] >= eval_start) & (targets['target_end'] <= eval_end)
    eval_origins = targets.index[m_eval]
    
    taus = {}
    audits = {}
    warns = {}
    
    # Online threshold selection ONLY for in-eval origins (fast & causally sound)
    for orig in eval_origins:
        tau, audit = capacity_threshold(
            score_meta=score_df, raw=raw, current_origin=orig,
            budget=budget, lookback_days=lookback_days
        )
        taus[orig] = tau
        audits[orig] = audit
        sc = score_df.loc[orig, 'score']
        warns[orig] = bool(np.isfinite(sc) and sc >= tau)
        
    # Construct aligned warning series across all targets.index
    warn_series = pd.Series([warns.get(orig, False) for orig in targets.index], index=targets.index)
    
    warning_df = score_df.copy()
    warning_df['tau'] = [taus.get(o, np.inf) for o in warning_df.index]
    warning_df['warn'] = [warns.get(o, False) for o in warning_df.index]
    
    # Interval metrics across unified eval_span (denominator 879/35 strictly verified)
    summary_metrics = interval_metrics(raw, targets, warn_series, eval_start, eval_end)
    
    # Fold metrics across disjoint sub-spans (warnings can cross inner boundaries)
    fold_metrics = []
    if disjoint_spans is not None:
        for idx, (s, e) in enumerate(disjoint_spans):
            f_met = interval_metrics(raw, targets, warn_series, s, e)
            f_met['fold_idx'] = idx
            f_met['start'] = s
            f_met['end'] = e
            fold_metrics.append(f_met)
            
    return warning_df, summary_metrics, fold_metrics

def qc_stream(raw, candidate, config):
    """
    Secondary QC triage chronological stream across individual raw rows.
    Daily refit using labels available <= t (timestamp + 6h <= t). Current row Fail label NEVER seen before score.
    """
    score_records = []
    fit_trace = []
    last_state = None
    last_fit_time = None
    
    delay_h = config.get('delay_h', 6)
    rolling_days = config.get('rolling_days', 56)
    refit_cadence_h = config.get('refit_cadence_h', 24)
    min_train_rows = config.get('min_train_rows', 100)
    
    sensor_cols = [c for c in raw.columns if c.startswith('sensor_')]
    
    for idx, r in raw.iterrows():
        t = pd.Timestamp(r['timestamp'])
        rid = r['raw_row_id']
        
        # Check refit condition (last_state is None checked BEFORE date delta to avoid overflow)
        refit_due = (last_state is None) or ((t - last_fit_time).total_seconds() >= refit_cadence_h * 3600.0)
        
        if refit_due:
            t_start = t - pd.Timedelta(days=rolling_days)
            m_eligible = (raw['timestamp'] >= t_start) & ((raw['timestamp'] + pd.Timedelta(hours=delay_h)) <= t)
            eligible_raw = raw.loc[m_eligible]
            
            # Assert label availability
            if len(eligible_raw) > 0:
                assert (eligible_raw['timestamp'] + pd.Timedelta(hours=delay_h) <= t).all(), f'QC label leak at {t}'
                
            n_fail = int(eligible_raw['fail'].sum()) if len(eligible_raw) > 0 else 0
            n_pass = len(eligible_raw) - n_fail
            
            if len(eligible_raw) >= min_train_rows and n_fail >= 5 and n_pass >= 5:
                if candidate == 'lr_qc_k24_half_odd':
                    pass_sub = eligible_raw[eligible_raw['fail'] == 0].iloc[::2]
                    fail_sub = eligible_raw[eligible_raw['fail'] == 1]
                    tr_subset = pd.concat([pass_sub, fail_sub]).sort_index()
                elif candidate == 'lr_qc_k24_half_rand':
                    pass_sub = eligible_raw[eligible_raw['fail'] == 0].sample(frac=0.5, random_state=42)
                    fail_sub = eligible_raw[eligible_raw['fail'] == 1]
                    tr_subset = pd.concat([pass_sub, fail_sub]).sort_index()
                else:
                    tr_subset = eligible_raw
                    
                actual_label_avail_max = (tr_subset['timestamp'] + pd.Timedelta(hours=delay_h)).max()
                assert actual_label_avail_max <= t, f'QC label leak in tr_subset at {t}'
                
                X_tr = tr_subset[sensor_cols]
                y_tr = tr_subset['fail'].astype(int)
                
                state = fit_model_state(
                    raw=raw, X_train=X_tr, y_train=y_tr, candidate=candidate,
                    task_type='qc', fit_time=t, delay_h=delay_h
                )
                state['last_train_label_available'] = actual_label_avail_max
                last_state = state
                last_fit_time = t
                fit_trace.append({
                    'fit_time': t, 'train_n': len(tr_subset),
                    'train_fail': n_fail, 'last_label_avail': actual_label_avail_max, 'status': 'fit'
                })
            else:
                past_fail_rate = (n_fail / len(eligible_raw)) if len(eligible_raw) > 0 else 0.066
                actual_label_avail_max = (eligible_raw['timestamp'] + pd.Timedelta(hours=delay_h)).max() if len(eligible_raw) > 0 else pd.NaT
                last_state = {
                    'candidate': candidate, 'type': 'fallback', 'fallback_val': past_fail_rate,
                    'fit_time': t, 'last_train_label_available': actual_label_avail_max,
                    'n_train': len(eligible_raw)
                }
                last_fit_time = t
                fit_trace.append({
                    'fit_time': t, 'train_n': len(eligible_raw),
                    'train_fail': n_fail, 'last_label_avail': actual_label_avail_max, 'status': 'fallback'
                })

        # Predict current raw row
        X_cur = raw.loc[[idx], sensor_cols]
        sc = float(predict_score(last_state, X_cur).iloc[0])
        score_records.append({
            'timestamp': t,
            'raw_row_id': rid,
            'score': sc,
            'fit_time': last_fit_time,
            'last_train_label_available': last_state.get('last_train_label_available'),
            'train_n': last_state.get('n_train', 0)
        })
        
    score_df = pd.DataFrame(score_records)
    return score_df, last_state, fit_trace

def apply_qc_policy(score_df, raw, budget, lookback_days=14, report_spans=None, threshold_budget=None):
    """
    Simulate online secondary QC triage under calendar report blocks and quota ceiling.
    Quota is evaluated sequentially on arriving prefix: warned_after <= floor(budget * seen).
    Hard prefix assertion enforced: warned_after <= floor(budget * seen) at EVERY row.
    """
    if threshold_budget is None:
        threshold_budget = budget
    def find_span(ts):
        if report_spans is None:
            return -1
        for idx, (s, e) in enumerate(report_spans):
            if s <= ts < e:
                return idx
        return -1

    current_span = -1
    seen_in_span = 0
    warned_in_span = 0
    
    rows_res = []
    taus = []
    
    for idx, r in score_df.iterrows():
        ts = pd.Timestamp(r['timestamp'])
        s_idx = find_span(ts)
        
        # Reset quota when entering a newly declared calendar span
        if s_idx != current_span:
            current_span = s_idx
            seen_in_span = 0
            warned_in_span = 0
            
        seen_in_span += 1
        tau, audit = qc_threshold(score_df, ts, budget=threshold_budget, lookback_days=lookback_days)
        taus.append(tau)
        
        cand_high = bool(np.isfinite(r['score']) and r['score'] >= tau)
        quota = int(np.floor(budget * seen_in_span))
        
        if cand_high and (warned_in_span + 1 <= quota):
            final_warn = True
            warned_in_span += 1
            quota_suppressed = False
        else:
            final_warn = False
            quota_suppressed = cand_high
            
        # Hard prefix quota assertion
        assert warned_in_span <= quota, f'Prefix quota breach at row {idx}: warned={warned_in_span}, quota={quota}'
            
        rows_res.append({
            'timestamp': ts,
            'raw_row_id': r['raw_row_id'],
            'candidate_high': cand_high,
            'final_warn': final_warn,
            'quota_suppressed': quota_suppressed,
            'seen': seen_in_span,
            'quota': quota,
            'warned_after': warned_in_span,
            'span_idx': s_idx
        })
        
    warning_df = pd.DataFrame(rows_res)
    warning_df['score'] = score_df['score'].values
    warning_df['tau'] = taus
    
    # Explicitly reindex raw_eval by raw_row_id to match warning_df order perfectly
    in_eval = warning_df[warning_df['span_idx'] >= 0].copy()
    raw_eval = raw[raw['raw_row_id'].isin(in_eval['raw_row_id'])].copy()
    raw_eval = raw_eval.set_index('raw_row_id').loc[in_eval['raw_row_id']].reset_index()
    assert (raw_eval['raw_row_id'].values == in_eval['raw_row_id'].values).all(), 'Row ID order mismatch!'
    
    summary_metrics = row_metrics(raw_eval, in_eval['final_warn'].values)
    
    fold_metrics = []
    if report_spans is not None:
        for idx, (s, e) in enumerate(report_spans):
            sub_w = warning_df[warning_df['span_idx'] == idx].copy()
            sub_raw = raw[raw['raw_row_id'].isin(sub_w['raw_row_id'])].copy()
            sub_raw = sub_raw.set_index('raw_row_id').loc[sub_w['raw_row_id']].reset_index()
            f_m = row_metrics(sub_raw, sub_w['final_warn'].values)
            f_m['fold_idx'] = idx
            f_m['start'] = s
            f_m['end'] = e
            fold_metrics.append(f_m)
            
    return warning_df, summary_metrics, fold_metrics

In [4]:
# 5. Untagged Driver: Prospective Walk-Forward Evaluation & Policy Tuning on Development Set
t_driver_start = time.time()
print('================================================================================')
print('STAGE 2 DRIVER: PREQUENTIAL WALK-FORWARD EVALUATION ON DEVELOPMENT DATASET')
print('Strict dev cutoff: timestamp < 2008-10-02 20:54 (1,253 raw entities)')
print('Evaluation spans : 3 x 14-day calendar blocks (Aug 21 -> Sep 4 -> Sep 18 -> Oct 2)')
print('Fixed denominator: 879 entities, 35 Fail events across evaluation window')
print('================================================================================')

# 1. Ingest Raw Dataset & Enforce Strict Dev Cutoff
raw_all = load_raw()
dev_cutoff = pd.Timestamp('2008-10-02 20:54:00')
dev_raw = raw_all[raw_all['timestamp'] < dev_cutoff].copy().reset_index(drop=True)
print(f'[DATA] Total raw rows: {len(raw_all)}, Dev raw rows: {len(dev_raw)}, Dev Fail count: {int(dev_raw["fail"].sum())}')

# 2. Build Feature Matrix Across Complete 6h Origin Grid on Development Set
min_ts = dev_raw['timestamp'].min()
dev_max_ts = dev_raw['timestamp'].max()
start_origin = (min_ts + pd.Timedelta(hours=168)).ceil('6h')
end_origin = dev_max_ts.floor('6h')
origins_grid = pd.date_range(start=start_origin, end=end_origin, freq='6h')

print(f'[FEATURES] Generating causal features for {len(origins_grid)} origins [{start_origin} -> {end_origin}]...')
t0_feat = time.time()
X_dev, avail_meta_dev = make_features(dev_raw, origins=origins_grid, delay_h=6, history_h=168, lead_h=6)
print(f'[FEATURES] Completed in {time.time() - t0_feat:.2f}s. Feature shape: {X_dev.shape}')

# Availability assertion: every origin strictly observes only past data
assert (avail_meta_dev['max_feature_available'].dropna() <= avail_meta_dev.index).all(), 'Feature availability leak!'

# 3. Build Ground-Truth Target Windows for Forecast Horizons H in [6, 12, 24]
horizons = [6, 12, 24]
targets_by_H = {}
for H in horizons:
    t_df = build_targets(dev_raw, origins_grid, horizon_h=H, lead_h=6, delay_h=6, q=0.10, min_event_samples=5)
    targets_by_H[H] = t_df.set_index('origin', drop=False)

# 4. Evaluation Specifications & Reporting Spans
eval_spans = [
    (pd.Timestamp('2008-08-21 18:00:00'), pd.Timestamp('2008-09-04 18:00:00')),
    (pd.Timestamp('2008-09-04 18:00:00'), pd.Timestamp('2008-09-18 18:00:00')),
    (pd.Timestamp('2008-09-18 18:00:00'), pd.Timestamp('2008-10-02 18:00:00')),
]
global_eval_span = (pd.Timestamp('2008-08-21 18:00:00'), pd.Timestamp('2008-10-02 18:00:00'))

# Verify fixed evaluation denominator
raw_eval = dev_raw[(dev_raw['timestamp'] >= global_eval_span[0]) & (dev_raw['timestamp'] < global_eval_span[1])]
assert len(raw_eval) == 879, f'Expected 879 evaluation entities, got {len(raw_eval)}'
assert int(raw_eval['fail'].sum()) == 35, f'Expected 35 evaluation Fails, got {int(raw_eval["fail"].sum())}'
print(f'[VERIFICATION] Global evaluation denominator verified: N={len(raw_eval)}, Fail={int(raw_eval["fail"].sum())}')

# ==============================================================================
# BRANCH A: PRIMARY QUALITY-RISK FORECAST WALK-FORWARD EVALUATION
# ==============================================================================
forecast_candidates = [
    # 5 Baselines
    'never_alert', 'fail_rate24', 'fail_rate72', 'fail_rate168', 'past_qc_ewma24',
    # 6 Primary Learned Models
    'lr_history', 'lr_sensor16', 'ridge_history', 'ridge_sensor16', 'hgb_poisson24', 'et_rate24'
]
forecast_budgets = [0.10, 0.15, 0.20, 0.25, 0.30]
forecast_lookbacks = [7, 14]

forecast_results = []
forecast_streams = {}
forecast_last_states = {}
forecast_traces = {}

print('\n>>> STARTING BRANCH A: FORECAST STREAMS & POLICY GRID (11 candidates x 3 H x 10 policies = 330 evaluations)...', flush=True)
for H in horizons:
    t_h_start = time.time()
    targets_H = targets_by_H[H]
    print(f'\n--- Processing Horizon H={H}h ---', flush=True)
    
    for cand in forecast_candidates:
        t_c_start = time.time()
        stream_cfg = {'delay_h': 6, 'rolling_days': 56, 'refit_cadence_h': 24, 'min_train_origins': 40}
        score_df, last_st, trace = forecast_stream(dev_raw, X_dev, avail_meta_dev, targets_H, cand, stream_cfg)
        
        forecast_streams[(H, cand)] = score_df
        forecast_last_states[(H, cand)] = last_st
        forecast_traces[(H, cand)] = trace
        
        # Policy grid search: 5 budgets x 2 lookbacks
        for b in forecast_budgets:
            for lb in forecast_lookbacks:
                warn_df, summary_m, fold_m = apply_forecast_policy(
                    score_df=score_df, raw=dev_raw, targets=targets_H,
                    budget=b, lookback_days=lb, eval_span=global_eval_span,
                    disjoint_spans=eval_spans
                )
                
                # Check denominator preservation for all configs
                assert summary_m['total_n'] == 879, f'Summary total_n mismatch: {summary_m["total_n"]}'
                assert summary_m['total_fail'] == 35, f'Summary total_fail mismatch: {summary_m["total_fail"]}'
                
                fold_wl = [fm['workload'] for fm in fold_m]
                fold_violations = sum(w > b for w in fold_wl)
                
                forecast_results.append({
                    'branch': 'forecast',
                    'horizon': H,
                    'candidate': cand,
                    'budget': b,
                    'lookback': lb,
                    'coverage': summary_m['coverage'],
                    'covered_fail': summary_m['covered_fail'],
                    'total_fail': summary_m['total_fail'],
                    'workload': summary_m['workload'],
                    'covered_n': summary_m['covered_n'],
                    'total_n': summary_m['total_n'],
                    'precision_entity': summary_m['precision_entity'],
                    'time_under_alert': summary_m['time_under_alert'],
                    'event_precision': summary_m['event_precision'],
                    'event_recall': summary_m['event_recall'],
                    'sparse_high_count': summary_m['sparse_high_count'],
                    'false_high_events': summary_m['false_high_events'],
                    'fold_wl_0': fold_wl[0],
                    'fold_wl_1': fold_wl[1],
                    'fold_wl_2': fold_wl[2],
                    'fold_violations': fold_violations
                })
        print(f'H={H}h | Cand: {cand:16s} evaluated in {time.time() - t_c_start:.2f}s', flush=True)

df_forecast_res = pd.DataFrame(forecast_results)

# ==============================================================================
# BRANCH B: SECONDARY QC INSPECTION TRIAGE WALK-FORWARD EVALUATION
# ==============================================================================
qc_candidates = [
    'balanced_lr_k1', 'balanced_lr_k8', 'balanced_lr_k24', 'balanced_lr_k24_c01',
    'balanced_rf', 'balanced_hgb', 'balanced_et_k64',
    'lr_qc_k24_half_odd', 'lr_qc_k24_half_rand'
]
qc_budgets = [0.20, 0.25, 0.30]
qc_lookback = 14

qc_results = []
qc_streams = {}
qc_last_states = {}
qc_traces = {}

print('\n>>> STARTING BRANCH B: QC TRIAGE STREAMS & POLICY GRID (9 candidates x 3 budgets = 27 evaluations)...', flush=True)
for cand in qc_candidates:
    t_qc_start = time.time()
    qc_cfg = {'delay_h': 6, 'rolling_days': 56, 'refit_cadence_h': 24, 'min_train_rows': 100}
    score_df_qc, last_st_qc, trace_qc = qc_stream(dev_raw, cand, qc_cfg)
    
    qc_streams[cand] = score_df_qc
    qc_last_states[cand] = last_st_qc
    qc_traces[cand] = trace_qc
    
    for b in qc_budgets:
        warn_df_qc, summary_m_qc, fold_m_qc = apply_qc_policy(
            score_df=score_df_qc, raw=dev_raw, budget=b,
            lookback_days=qc_lookback, report_spans=eval_spans
        )
        fold_wl = [fm['workload'] for fm in fold_m_qc]
        qc_results.append({
            'branch': 'qc',
            'candidate': cand,
            'budget': b,
            'lookback': qc_lookback,
            'coverage': summary_m_qc['coverage'],
            'covered_fail': summary_m_qc['covered_fail'],
            'total_fail': summary_m_qc['total_fail'],
            'workload': summary_m_qc['workload'],
            'covered_n': summary_m_qc['covered_n'],
            'total_n': summary_m_qc['total_n'],
            'precision_entity': summary_m_qc['precision_entity'],
            'f1': summary_m_qc['F1'],
            'tp': summary_m_qc['TP'],
            'fp': summary_m_qc['FP'],
            'fn': summary_m_qc['FN'],
            'tn': summary_m_qc['TN'],
            'fold_wl_0': fold_wl[0],
            'fold_wl_1': fold_wl[1],
            'fold_wl_2': fold_wl[2]
        })
    print(f'QC Cand: {cand:22s} evaluated in {time.time() - t_qc_start:.2f}s', flush=True)

df_qc_res = pd.DataFrame(qc_results)

# ==============================================================================
# MODEL SELECTION & REPORTING (OPTIMIZING ON DEV ONLY)
# ==============================================================================
print('\n' + '='*80)
print('DEVELOPMENT MODEL SELECTION & COMPARATIVE BENCHMARKING')
print('Objective: Maximize Coverage subject to Aggregate Workload <= 0.30')
print('='*80)

# Branch A: Filter eligible configurations (Aggregate Workload <= 0.30)
eligible_A = df_forecast_res[df_forecast_res['workload'] <= 0.30].copy()
eligible_A = eligible_A.sort_values(by=['coverage', 'workload'], ascending=[False, True])

print('\n>>> TOP 12 ELIGIBLE FORECAST CONFIGURATIONS (Workload <= 0.30):')
cols_display_A = ['horizon', 'candidate', 'budget', 'lookback', 'coverage', 'covered_fail', 'workload', 'precision_entity', 'fold_violations']
print(eligible_A[cols_display_A].head(12).to_string(index=False))

# Select Forecast Winner
if len(eligible_A) > 0:
    winner_A = eligible_A.iloc[0].to_dict()
    meets_70_A = bool(winner_A['coverage'] >= 0.70)
else:
    winner_A = df_forecast_res.sort_values(by=['coverage'], ascending=False).iloc[0].to_dict()
    meets_70_A = False

print(f'\n[FORECAST WINNER]: H={winner_A["horizon"]}h | Candidate: {winner_A["candidate"]} | Budget: {winner_A["budget"]} | Lookback: {winner_A["lookback"]}d')
print(f'Coverage: {winner_A["coverage"]*100:.2f}% ({winner_A["covered_fail"]}/{winner_A["total_fail"]}) | Workload: {winner_A["workload"]*100:.2f}% | Target 70% Met: {meets_70_A}')

# Branch B: Filter eligible QC triage configurations
eligible_B = df_qc_res[df_qc_res['workload'] <= 0.30].copy()
eligible_B = eligible_B.sort_values(by=['coverage', 'workload'], ascending=[False, True])

print('\n>>> ALL QC TRIAGE CONFIGURATIONS:')
cols_display_B = ['candidate', 'budget', 'coverage', 'covered_fail', 'workload', 'precision_entity', 'f1']
print(df_qc_res[cols_display_B].to_string(index=False))

winner_B = eligible_B.iloc[0].to_dict() if len(eligible_B) > 0 else df_qc_res.iloc[0].to_dict()
meets_numeric_B = bool(winner_B['workload'] <= winner_B['budget'])
print(f'\n[QC TRIAGE WINNER]: Candidate: {winner_B["candidate"]} | Budget: {winner_B["budget"]} | Lookback: {winner_B["lookback"]}d')
print(f'Coverage: {winner_B["coverage"]*100:.2f}% ({winner_B["covered_fail"]}/{winner_B["total_fail"]}) | Workload: {winner_B["workload"]*100:.2f}% | Numeric Budget Met: {meets_numeric_B}')

# ==============================================================================
# ARTIFACT PERSISTENCE: Version 2/experiment.joblib
# ==============================================================================
h_secom = hashlib.sha256(open(PROJECT_ROOT / 'data' / 'secom' / 'secom.data', 'rb').read()).hexdigest()
h_labels = hashlib.sha256(open(PROJECT_ROOT / 'data' / 'secom' / 'secom_labels.data', 'rb').read()).hexdigest()

selected_key_A = (winner_A['horizon'], winner_A['candidate'])
selected_score_df_A = forecast_streams[selected_key_A].copy()
selected_last_state_A = forecast_last_states[selected_key_A]
selected_trace_A = forecast_traces[selected_key_A]

selected_cand_B = winner_B['candidate']
selected_score_df_B = qc_streams[selected_cand_B].copy()
selected_last_state_B = qc_last_states[selected_cand_B]
selected_trace_B = qc_traces[selected_cand_B]

artifact_data = {
    'protocol': {
        'lead': '6h',
        'delay': '6h',
        'history': '168h',
        'cadence': '6h',
        'horizons': horizons,
        'q': 0.10,
        'min_event_samples': 5,
        'dev_cutoff': str(dev_cutoff),
        'prior_holdout_known_exposure': True,
        'client_frozen_date': '2026-10-26'
    },
    'stream_configs': {
        'forecast': {'delay_h': 6, 'rolling_days': 56, 'refit_cadence_h': 24, 'min_train_origins': 40},
        'qc': {'delay_h': 6, 'rolling_days': 56, 'refit_cadence_h': 24, 'min_train_rows': 100}
    },
    'grids': {
        'forecast_candidates': forecast_candidates,
        'forecast_budgets': forecast_budgets,
        'forecast_lookbacks': forecast_lookbacks,
        'qc_candidates': qc_candidates,
        'qc_budgets': qc_budgets,
        'qc_lookback': qc_lookback
    },
    'raw_hashes': {
        'secom_data_sha256': h_secom,
        'secom_labels_sha256': h_labels
    },
    'temporal_bounds': {
        'dev_start': str(min_ts),
        'dev_cutoff': str(dev_cutoff),
        'global_eval_start': str(global_eval_span[0]),
        'global_eval_end': str(global_eval_span[1]),
        'eval_spans': [(str(s), str(e)) for s, e in eval_spans],
        'test_A_start': '2008-10-03 06:00:00',
        'test_A_end': '2008-10-17 06:00:00',
        'test_B_start': str(dev_cutoff),
        'test_A_expected_n': 297,
        'test_A_expected_fail': 16,
        'test_B_expected_n': 314,
        'test_B_expected_fail': 17
    },
    'forecast_summary': df_forecast_res,
    'qc_summary': df_qc_res,
    'selection': {
        'winner_A': winner_A,
        'winner_B': winner_B,
        'meets_70_A': meets_70_A,
        'meets_numeric_B': meets_numeric_B
    },
    'selected_dev_scores': {
        'branch_A': selected_score_df_A,
        'branch_B': selected_score_df_B
    },
    'selected_last_states': {
        'branch_A': selected_last_state_A,
        'branch_B': selected_last_state_B
    },
    'selected_traces': {
        'branch_A': selected_trace_A,
        'branch_B': selected_trace_B
    },
    'versions': {
        'python': sys.version.split()[0],
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'sklearn': sklearn.__version__,
        'joblib': joblib.__version__
    }
}

artifact_path = VERSION_DIR / 'experiment.joblib'
joblib.dump(artifact_data, artifact_path)
print(f'\n[ARTIFACT] Successfully saved complete experimental state to {artifact_path}')
print(f'[STAGE 2 COMPLETE] Total runtime: {time.time() - t_driver_start:.2f}s')

STAGE 2 DRIVER: PREQUENTIAL WALK-FORWARD EVALUATION ON DEVELOPMENT DATASET
Strict dev cutoff: timestamp < 2008-10-02 20:54 (1,253 raw entities)
Evaluation spans : 3 x 14-day calendar blocks (Aug 21 -> Sep 4 -> Sep 18 -> Oct 2)
Fixed denominator: 879 entities, 35 Fail events across evaluation window


[DATA] Total raw rows: 1567, Dev raw rows: 1253, Dev Fail count: 87
[FEATURES] Generating causal features for 274 origins [2008-07-26 12:00:00 -> 2008-10-02 18:00:00]...


[FEATURES] Completed in 8.02s. Feature shape: (274, 3564)


[VERIFICATION] Global evaluation denominator verified: N=879, Fail=35

>>> STARTING BRANCH A: FORECAST STREAMS & POLICY GRID (11 candidates x 3 H x 10 policies = 330 evaluations)...



--- Processing Horizon H=6h ---


H=6h | Cand: never_alert      evaluated in 3.76s


... (41 outputs left out)

QC Cand: balanced_et_k64        evaluated in 58.94s


QC Cand: lr_qc_k24_half_odd     evaluated in 13.31s


QC Cand: lr_qc_k24_half_rand    evaluated in 13.37s



DEVELOPMENT MODEL SELECTION & COMPARATIVE BENCHMARKING
Objective: Maximize Coverage subject to Aggregate Workload <= 0.30

>>> TOP 12 ELIGIBLE FORECAST CONFIGURATIONS (Workload <= 0.30):
 horizon      candidate  budget  lookback  coverage  covered_fail  workload  precision_entity  fold_violations
       6  ridge_history    0.30         7  0.342857            12  0.251422          0.054299                1
      12     lr_history    0.25         7  0.342857            12  0.296928          0.045977                2
       6  ridge_history    0.25         7  0.314286            11  0.221843          0.056410                2
      24  ridge_history    0.30         7  0.314286            11  0.281001          0.044534                2
      24   fail_rate168    0.30         7  0.285714            10  0.154721          0.073529                1
      24 ridge_sensor16    0.20         7  0.257143             9  0.114903          0.089109                1
      24 ridge_sensor16    0.25    

In [1]:
import os
import sys
import json
import time
import copy
import hashlib
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

# 1. Environment Bootstrap & Core Directory Setup
def find_project_root(start_path=None):
    p = Path(start_path).resolve() if start_path else Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if (parent / 'AGENTS.md').exists() and (parent / 'data' / 'secom').exists():
            return parent
    raise RuntimeError(f'Could not locate project root from {p}')

PROJECT_ROOT = find_project_root()
VERSION_DIR = PROJECT_ROOT / 'Version 2'
DATA_DIR = PROJECT_ROOT / 'data' / 'secom'
artifact_path = VERSION_DIR / 'experiment.joblib'

print('================================================================================')
print('TAGGED EXTENSION: POST-TRAINING POLICY REFINEMENT (v2_policy_refinement)')
print('Runtime bootstrap from 03 tagged helpers & frozen experiment.joblib scores')
print('Strictly NO model refitting, NO new feature training, NO holdout opening')
print('================================================================================')

# 2. Ingest 03_Train Tagged Helpers into Runtime
train_nb_path = VERSION_DIR / '03_Train.ipynb'
assert train_nb_path.exists(), f'Missing 03_Train notebook at {train_nb_path}'

with open(train_nb_path, 'r', encoding='utf-8') as f:
    nb_train = json.load(f)

helper_blocks = []
for cell in nb_train.get('cells', []):
    if cell.get('cell_type') == 'code':
        tags = cell.get('metadata', {}).get('tags', [])
        if 'v2_training_helpers' in tags:
            helper_blocks.append(''.join(cell.get('source', [])))

assert len(helper_blocks) > 0, 'No code cells tagged with v2_training_helpers found in 03_Train.ipynb!'
exec('\n\n'.join(helper_blocks), globals())

required_fns = ['load_raw', 'build_targets', 'apply_forecast_policy', 'apply_qc_policy', 'capacity_threshold', 'qc_threshold']
for fn in required_fns:
    assert fn in globals(), f'Helper function {fn} missing from runtime!'
print(f'[BOOTSTRAP] Successfully loaded {len(helper_blocks)} helper blocks from 03_Train.ipynb')

# 3. Load Frozen Artifact & Deepcopy Stage 1 Selection
assert artifact_path.exists(), f'Missing artifact at {artifact_path}'
exp = joblib.load(artifact_path)

stage1_selection = copy.deepcopy(exp['selection'])
print(f'[ARTIFACT] Frozen Phase 1 Selection: Winner A={stage1_selection["winner_A"]["candidate"]} (H={stage1_selection["winner_A"]["horizon"]}h, budget={stage1_selection["winner_A"]["budget"]}, lb={stage1_selection["winner_A"]["lookback"]}d) | Winner B={stage1_selection["winner_B"]["candidate"]} (budget={stage1_selection["winner_B"]["budget"]}, lb={stage1_selection["winner_B"]["lookback"]}d)')

# 4. Verify Raw Data Hash & Dev Cutoff
raw_all = load_raw()
data_file = DATA_DIR / 'secom.data'
labels_file = DATA_DIR / 'secom_labels.data'
sha_data = hashlib.sha256(data_file.read_bytes()).hexdigest()
sha_labels = hashlib.sha256(labels_file.read_bytes()).hexdigest()

assert exp['raw_hashes']['secom_data_sha256'] == sha_data, 'SECOM data hash mismatch!'
assert exp['raw_hashes']['secom_labels_sha256'] == sha_labels, 'SECOM labels hash mismatch!'

dev_cutoff = pd.Timestamp(exp['protocol']['dev_cutoff'])
dev_raw = raw_all[raw_all['timestamp'] < dev_cutoff].copy().reset_index(drop=True)
assert len(dev_raw) == 1253, f'Dev raw count mismatch: expected 1253, got {len(dev_raw)}'
assert int(dev_raw['fail'].sum()) == 87, f'Dev raw fail mismatch: expected 87, got {int(dev_raw["fail"].sum())}'
print(f'[VERIFY] Hashes and Dev cutoff ({dev_cutoff}) verified. Total dev rows: {len(dev_raw)}, Fails: {int(dev_raw["fail"].sum())}')

# 5. Fixed Evaluation Setup across 3 Calendar Folds (Denominator 879/35)
eval_spans = [
    (pd.Timestamp('2008-08-21 18:00:00'), pd.Timestamp('2008-09-04 18:00:00')),
    (pd.Timestamp('2008-09-04 18:00:00'), pd.Timestamp('2008-09-18 18:00:00')),
    (pd.Timestamp('2008-09-18 18:00:00'), pd.Timestamp('2008-10-02 18:00:00')),
]
global_eval_span = (pd.Timestamp('2008-08-21 18:00:00'), pd.Timestamp('2008-10-02 18:00:00'))

# 6. Branch A Policy Refinement (Cached Ridge History H=6 OOS Scores)
print('\n--- Branch A Policy Refinement: Ridge History (H=6) ---')
scores_A = exp['selected_dev_scores']['branch_A'].copy()
assert stage1_selection['winner_A']['horizon'] == 6, 'Phase 1 winner A horizon mismatch!'
assert stage1_selection['winner_A']['candidate'] == 'ridge_history', 'Phase 1 winner A candidate mismatch!'

targets_H6 = build_targets(dev_raw, scores_A.index, horizon_h=6, lead_h=6, delay_h=6, q=0.10, min_event_samples=5).set_index('origin', drop=False)

budgets_A = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
lookbacks_A = [7, 14]

refined_rows_A = []
for b in budgets_A:
    for lb in lookbacks_A:
        w_df, summ, folds = apply_forecast_policy(
            scores_A, dev_raw, targets_H6,
            budget=b, lookback_days=lb,
            eval_span=global_eval_span, disjoint_spans=eval_spans
        )
        assert summ['total_n'] == 879 and summ['total_fail'] == 35, f'A denom mismatch at b={b}, lb={lb}'
        
        row_dict = {
            'phase': 'refined',
            'horizon': 6,
            'candidate': 'ridge_history',
            'budget': b,
            'lookback': lb,
            'coverage': summ['coverage'],
            'covered_fail': summ['covered_fail'],
            'total_fail': summ['total_fail'],
            'workload': summ['workload'],
            'covered_n': summ['covered_n'],
            'total_n': summ['total_n'],
            'precision_entity': summ['precision_entity'],
            'time_under_alert': summ.get('time_under_alert', np.nan),
            'event_precision': summ.get('event_precision', np.nan),
            'event_recall': summ.get('event_recall', np.nan),
            'sparse_high_count': summ.get('sparse_high_count', np.nan),
            'false_high_events': summ.get('false_high_events', np.nan),
            'fold_wl_0': folds[0]['workload'],
            'fold_wl_1': folds[1]['workload'],
            'fold_wl_2': folds[2]['workload'],
            'fold_violations': sum(1 for f in folds if f['workload'] > b),
            'eligible_actual_30': bool(summ['workload'] <= 0.30)
        }
        refined_rows_A.append(row_dict)

df_refined_A = pd.DataFrame(refined_rows_A)

# Select Winner A among eligible (actual aggregate workload <= 0.30)
eligible_A = df_refined_A[df_refined_A['eligible_actual_30']].copy()
eligible_A = eligible_A.sort_values(by=['coverage', 'workload'], ascending=[False, True])
winner_A = eligible_A.iloc[0].to_dict()

# Assertions for Winner A candidate and horizon integrity
assert winner_A['candidate'] == stage1_selection['winner_A']['candidate'], 'Branch A candidate changed!'
assert winner_A['horizon'] == stage1_selection['winner_A']['horizon'], 'Branch A horizon changed!'

# Retain original 330 table and append refined rows
orig_forecast_summary = exp['forecast_summary'].copy()
if 'phase' not in orig_forecast_summary.columns:
    orig_forecast_summary['phase'] = 'original'
if 'eligible_actual_30' not in orig_forecast_summary.columns:
    orig_forecast_summary['eligible_actual_30'] = orig_forecast_summary['workload'] <= 0.30

combined_forecast_summary = pd.concat([orig_forecast_summary, df_refined_A], ignore_index=True)

# 7. Branch B Policy Refinement (Cached Balanced HGB OOS Scores)
print('\n--- Branch B Policy Refinement: Balanced HGB ---')
scores_B = exp['selected_dev_scores']['branch_B'].copy()
assert stage1_selection['winner_B']['candidate'] == 'balanced_hgb', 'Phase 1 winner B candidate mismatch!'

budgets_B = [0.20, 0.25, 0.30]
th_budgets_B = [0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60, 0.75, 1.00]

refined_rows_B = []
for b in budgets_B:
    for tb in th_budgets_B:
        w_df, summ, folds = apply_qc_policy(
            scores_B, dev_raw,
            budget=b, lookback_days=14,
            report_spans=eval_spans,
            threshold_budget=tb
        )
        assert summ['total_n'] == 879 and summ['total_fail'] == 35, f'B denom mismatch at b={b}, tb={tb}'
        
        row_dict = {
            'phase': 'refined',
            'candidate': 'balanced_hgb',
            'budget': b,
            'threshold_budget': tb,
            'lookback': 14,
            'coverage': summ['coverage'],
            'covered_fail': summ['covered_fail'],
            'total_fail': summ['total_fail'],
            'workload': summ['workload'],
            'covered_n': summ['covered_n'],
            'total_n': summ['total_n'],
            'precision_entity': summ['precision_entity'],
            'f1': summ.get('F1', np.nan),
            'tp': summ.get('TP', summ['covered_fail']),
            'fp': summ.get('FP', summ['covered_n'] - summ['covered_fail']),
            'fn': summ.get('FN', summ['total_fail'] - summ['covered_fail']),
            'tn': summ.get('TN', summ['total_n'] - summ['covered_n']),
            'fold_wl_0': folds[0]['workload'],
            'fold_wl_1': folds[1]['workload'],
            'fold_wl_2': folds[2]['workload'],
            'eligible_actual_30': bool(summ['workload'] <= 0.30)
        }
        refined_rows_B.append(row_dict)

df_refined_B = pd.DataFrame(refined_rows_B)

# Select Winner B among eligible (actual aggregate workload <= 0.30)
eligible_B = df_refined_B[df_refined_B['eligible_actual_30']].copy()
eligible_B = eligible_B.sort_values(by=['coverage', 'workload'], ascending=[False, True])
winner_B = eligible_B.iloc[0].to_dict()

# Assertions for Winner B candidate integrity
assert winner_B['candidate'] == stage1_selection['winner_B']['candidate'], 'Branch B candidate changed!'

# Retain original 27 table, fill threshold_budget=budget, and append refined rows
orig_qc_summary = exp['qc_summary'].copy()
if 'phase' not in orig_qc_summary.columns:
    orig_qc_summary['phase'] = 'original'
if 'threshold_budget' not in orig_qc_summary.columns:
    orig_qc_summary['threshold_budget'] = orig_qc_summary['budget']
if 'eligible_actual_30' not in orig_qc_summary.columns:
    orig_qc_summary['eligible_actual_30'] = orig_qc_summary['workload'] <= 0.30

combined_qc_summary = pd.concat([orig_qc_summary, df_refined_B], ignore_index=True)

# 8. Print Concise Policy Tables and Winners
print('\n================================================================================')
print('BRANCH A REFINED POLICY GRID (18 EVALUATIONS):')
print(df_refined_A[['budget', 'lookback', 'coverage', 'covered_fail', 'workload', 'covered_n', 'precision_entity', 'eligible_actual_30']].to_string(index=False))

print('\nBRANCH B REFINED POLICY GRID (27 EVALUATIONS):')
print(df_refined_B[['budget', 'threshold_budget', 'coverage', 'covered_fail', 'workload', 'covered_n', 'precision_entity', 'eligible_actual_30']].to_string(index=False))

print('\n================================================================================')
print(f'[WINNER A (SELECTED)]: Candidate={winner_A["candidate"]}, H={winner_A["horizon"]}h, Budget={winner_A["budget"]}, Lookback={winner_A["lookback"]}d')
print(f'  Coverage: {winner_A["coverage"]*100:.2f}% ({winner_A["covered_fail"]}/{winner_A["total_fail"]}), Workload: {winner_A["workload"]*100:.2f}% ({winner_A["covered_n"]}/{winner_A["total_n"]})')
print(f'  Fold Workloads: [{winner_A["fold_wl_0"]*100:.2f}%, {winner_A["fold_wl_1"]*100:.2f}%, {winner_A["fold_wl_2"]*100:.2f}%]')

print(f'\n[WINNER B (SELECTED)]: Candidate={winner_B["candidate"]}, QuotaBudget={winner_B["budget"]}, ThresholdBudget={winner_B["threshold_budget"]}, Lookback={winner_B["lookback"]}d')
print(f'  Coverage: {winner_B["coverage"]*100:.2f}% ({winner_B["covered_fail"]}/{winner_B["total_fail"]}), Workload: {winner_B["workload"]*100:.2f}% ({winner_B["covered_n"]}/{winner_B["total_n"]})')
print(f'  Fold Workloads: [{winner_B["fold_wl_0"]*100:.2f}%, {winner_B["fold_wl_1"]*100:.2f}%, {winner_B["fold_wl_2"]*100:.2f}%]')
print('================================================================================')

# 9. Update Artifact Preserving Historical Objects
meets_70_A = bool((winner_A['coverage'] >= 0.70) and (winner_A['workload'] <= 0.30))
meets_70_B = bool((winner_B['coverage'] >= 0.70) and (winner_B['workload'] <= 0.30))
old_meets_numeric_B = bool(winner_B['workload'] <= winner_B['budget'])

exp['forecast_summary'] = combined_forecast_summary
exp['qc_summary'] = combined_qc_summary

exp['selection']['winner_A'] = winner_A
exp['selection']['winner_B'] = winner_B
exp['selection']['meets_70_A'] = meets_70_A
exp['selection']['meets_70_B'] = meets_70_B
exp['selection']['meets_numeric_B'] = old_meets_numeric_B

exp['policy_refinement_v2'] = {
    'stage1_selection': stage1_selection,
    'grid_A': {'budgets': budgets_A, 'lookbacks': lookbacks_A},
    'grid_B': {'budgets': budgets_B, 'threshold_budgets': th_budgets_B, 'lookback': 14},
    'refined_table_A': df_refined_A,
    'refined_table_B': df_refined_B,
    'winner_A': winner_A,
    'winner_B': winner_B,
    'scope_caveat': 'Policy refinement evaluated on frozen Stage 1 OOS scores without refitting or holdout exposure.'
}

# Preserve scores, states, traces untouched
assert 'selected_dev_scores' in exp, 'Missing selected_dev_scores'
assert 'selected_last_states' in exp, 'Missing selected_last_states'
assert 'selected_traces' in exp, 'Missing selected_traces'

joblib.dump(exp, artifact_path)
print(f'[SUCCESS] Artifact updated at {artifact_path} with policy_refinement_v2 and updated selection.')


TAGGED EXTENSION: POST-TRAINING POLICY REFINEMENT (v2_policy_refinement)
Runtime bootstrap from 03 tagged helpers & frozen experiment.joblib scores
Strictly NO model refitting, NO new feature training, NO holdout opening


[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2
[BOOTSTRAP] PROJECT_ROOT: D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM
[BOOTSTRAP] VERSION_DIR:  D:\AI_Bussiness_Huflit\Industrial_AI_on_UCISECOM\Version 2
[BOOTSTRAP] Successfully imported 5 definition code cells from 02_Preprocessing.ipynb
[BOOTSTRAP] Available functions: load_raw, make_features, build_targets, training_maturity_gate, interval_metrics, row_metrics, capacity_threshold, qc_threshold
[BOOTSTRAP] Successfully loaded 3 helper blocks from 03_Train.ipynb
[ARTIFACT] Frozen Phase 1 Selection: Winner A=ridge_history (H=6h, budget=0.3, lb=7d) | Winner B=balanced_hgb (budget=0.3, lb=14d)


[VERIFY] Hashes and Dev cutoff (2008-10-02 20:54:00) verified. Total dev rows: 1253, Fails: 87

--- Branch A Policy Refinement: Ridge History (H=6) ---



--- Branch B Policy Refinement: Balanced HGB ---



BRANCH A REFINED POLICY GRID (18 EVALUATIONS):
 budget  lookback  coverage  covered_fail  workload  covered_n  precision_entity  eligible_actual_30
   0.10         7  0.085714             3  0.077361         68          0.044118                True
   0.10        14  0.000000             0  0.006826          6          0.000000                True
   0.15         7  0.114286             4  0.098976         87          0.045977                True
   0.15        14  0.000000             0  0.006826          6          0.000000                True
   0.20         7  0.228571             8  0.203641        179          0.044693                True
   0.20        14  0.057143             2  0.070535         62          0.032258                True
   0.25         7  0.314286            11  0.221843        195          0.056410                True
   0.25        14  0.114286             4  0.118316        104          0.038462                True
   0.30         7  0.342857            12  